# 03 — Data Preprocessing

Notebook này tạo các bảng dẫn xuất từ CSV gốc sau khi đã xem kết quả EDA. File gốc không bị ghi đè.

**Chính sách:** không tạo doanh số cho bốn ngày 25/12 vắng trong train; không điền dầu; không loại outlier/series nhiều số 0. Transactions cùng ngày không được nhập thành feature dự báo vì có thể gây leakage; lịch sử giao dịch được lưu riêng cho bước tạo lag hoặc dự báo transactions.


## 1. Import thư viện và tìm project root

Dữ liệu đã xử lý sẽ vào `data/processed/`; các báo cáo kiểm tra vào `reports/preprocessing/`.


In [1]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns",100)
pd.set_option("display.width",160)


In [2]:
candidates=[Path.cwd(),*Path.cwd().parents]
ROOT=next((p for p in candidates if (p/"data"/"raw"/"train.csv").is_file()),None)
if ROOT is None: raise FileNotFoundError("Không tìm thấy data/raw/train.csv. Mở notebook từ repo hoặc thư mục con.")
RAW_DIR=ROOT/"data"/"raw"
PROCESSED_DIR=ROOT/"data"/"processed"
REPORT_DIR=ROOT/"reports"/"preprocessing"
PROCESSED_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print("Project root:",ROOT)
print("Processed data:",PROCESSED_DIR)
print("Reports:",REPORT_DIR)


Project root: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-
Processed data: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\data\processed
Reports: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\preprocessing


## 2. Đọc CSV gốc và kiểm tra khóa nghiệp vụ

Chỉ đọc nguồn; các bước sau tạo DataFrame mới. Dừng nếu khóa không duy nhất để tránh nhân dòng khi merge.


In [3]:
train=pd.read_csv(RAW_DIR/"train.csv",parse_dates=["date"],low_memory=False)
test=pd.read_csv(RAW_DIR/"test.csv",parse_dates=["date"],low_memory=False)
stores=pd.read_csv(RAW_DIR/"stores.csv",low_memory=False)
oil=pd.read_csv(RAW_DIR/"oil.csv",parse_dates=["date"],low_memory=False)
holidays=pd.read_csv(RAW_DIR/"holidays_events.csv",parse_dates=["date"],low_memory=False)
transactions=pd.read_csv(RAW_DIR/"transactions.csv",parse_dates=["date"],low_memory=False)
assert not train.duplicated(["date","store_nbr","family"]).any(),"Duplicate train key"
assert not test.duplicated(["date","store_nbr","family"]).any(),"Duplicate test key"
assert stores.store_nbr.is_unique,"Duplicate store metadata key"
assert oil.date.is_unique,"Duplicate oil date key"
assert not transactions.duplicated(["date","store_nbr"]).any(),"Duplicate transaction key"
print({"train":train.shape,"test":test.shape,"stores":stores.shape,"oil":oil.shape,"holidays":holidays.shape,"transactions":transactions.shape})


{'train': (3000888, 6), 'test': (28512, 5), 'stores': (54, 5), 'oil': (1218, 2), 'holidays': (350, 6), 'transactions': (83488, 3)}


## 3. Kiểm tra calendar và Store-Family coverage

So sánh mỗi series với các ngày train thực sự quan sát được. Báo cáo bốn ngày vắng nhưng không tạo thêm dòng hay `sales = 0`.


In [4]:
calendar_days=pd.date_range(train.date.min(),train.date.max(),freq="D")
observed_dates=pd.DatetimeIndex(train.date.unique()).sort_values()
missing_train_dates=calendar_days.difference(observed_dates)
series_n=train[["store_nbr","family"]].drop_duplicates().shape[0]
series_date_check=train.groupby(["store_nbr","family"]).date.agg(first_date="min",last_date="max",n_dates="nunique").reset_index()
incomplete_series=series_date_check[
    series_date_check.first_date.ne(train.date.min())
    | series_date_check.last_date.ne(train.date.max())
    | series_date_check.n_dates.ne(train.date.nunique())]
series_per_date=train.groupby("date").size()
date_coverage_report=pd.DataFrame([{
    "calendar_days_in_range":len(calendar_days),"observed_train_dates":train.date.nunique(),
    "missing_calendar_dates":len(missing_train_dates),"missing_dates":", ".join(str(d.date()) for d in missing_train_dates),
    "store_family_series":series_n,"incomplete_series_on_observed_dates":len(incomplete_series),
    "dates_with_unexpected_series_count":int(series_per_date.ne(series_n).sum())}])
display(date_coverage_report); display(incomplete_series.head())


,calendar_days_in_range,observed_train_dates,missing_calendar_dates,missing_dates,store_family_series,incomplete_series_on_observed_dates,dates_with_unexpected_series_count
0,1688,1684,4,"2013-12-25, 2014-12-25, 2015-12-25, 2016-12-25",1782,0,0


,store_nbr,family,first_date,last_date,n_dates


## 4. Ghép metadata cửa hàng

`stores.csv` có một dòng mỗi `store_nbr`, nên dùng `validate="many_to_one"`. Kiểm tra row count và metadata không khớp.


In [5]:
train_base=train.copy(); test_base=test.copy()
train_rows_before=len(train_base); test_rows_before=len(test_base)
train_base=train_base.merge(stores,on="store_nbr",how="left",validate="many_to_one",sort=False)
test_base=test_base.merge(stores,on="store_nbr",how="left",validate="many_to_one",sort=False)
assert len(train_base)==train_rows_before and len(test_base)==test_rows_before
store_cols=["city","state","type","cluster"]
assert not train_base[store_cols].isna().any().any(),"Unmatched train store metadata"
assert not test_base[store_cols].isna().any().any(),"Unmatched test store metadata"
print("Store merge preserved row counts:",len(train_base),len(test_base))


Store merge preserved row counts: 3000888 28512


## 5. Ghép giá dầu, giữ nguyên missing

Cờ `oil_date_record_present` phân biệt ngày không có dòng nguồn với ngày có dòng nhưng giá dầu NaN. Không dùng backward fill hoặc giá tương lai để lấp missing.


In [6]:
oil_for_merge=oil.copy(); oil_for_merge["oil_date_record_present"]=1
train_base=train_base.merge(oil_for_merge,on="date",how="left",validate="many_to_one",sort=False)
test_base=test_base.merge(oil_for_merge,on="date",how="left",validate="many_to_one",sort=False)
for frame in (train_base,test_base): frame["oil_date_record_present"]=frame.oil_date_record_present.fillna(0).astype("int8")
assert len(train_base)==train_rows_before and len(test_base)==test_rows_before
oil_coverage_report=pd.DataFrame([
 {"split":"train","rows":len(train_base),"missing_oil_values":int(train_base.dcoilwtico.isna().sum()),"dates_without_source_record":int(train_base.loc[train_base.oil_date_record_present.eq(0),"date"].nunique()),"source_record_price_missing_dates":int(train_base.loc[train_base.oil_date_record_present.eq(1)&train_base.dcoilwtico.isna(),"date"].nunique())},
 {"split":"test","rows":len(test_base),"missing_oil_values":int(test_base.dcoilwtico.isna().sum()),"dates_without_source_record":int(test_base.loc[test_base.oil_date_record_present.eq(0),"date"].nunique()),"source_record_price_missing_dates":int(test_base.loc[test_base.oil_date_record_present.eq(1)&test_base.dcoilwtico.isna(),"date"].nunique())}])
display(oil_coverage_report)


,split,rows,missing_oil_values,dates_without_source_record,source_record_price_missing_dates
0,train,3000888,928422,481,40
1,test,28512,7128,4,0


## 6. Tạo holiday features ở Date-Store

National áp dụng cho mọi cửa hàng; Regional khớp `locale_name` với `state`; Local khớp với `city`. Ngày gốc có `transferred=True` không được tính là ngày nghỉ hoạt động. Ngày tổ chức thực tế được nhận diện bằng `type="Transfer"`. Event và Work Day có biến riêng. Các biến `matched_*_record_count` chỉ đếm bản ghi nguồn khớp; chúng không mặc định là ngày nghỉ.


In [7]:
store_lookup=stores[["store_nbr","city","state"]].copy()
national=holidays.loc[holidays.locale.eq("National")].merge(store_lookup,how="cross")
regional=holidays.loc[holidays.locale.eq("Regional")].merge(store_lookup,left_on="locale_name",right_on="state",how="inner",validate="many_to_many")
local=holidays.loc[holidays.locale.eq("Local")].merge(store_lookup,left_on="locale_name",right_on="city",how="inner",validate="many_to_many")
holiday_matches=pd.concat([national,regional,local],ignore_index=True)
holiday_matches["is_transferred_original"]=holiday_matches.transferred.fillna(False).astype(bool)
holiday_matches["is_transfer_date"]=holiday_matches.type.eq("Transfer")
holiday_matches["is_active_holiday"]=(~holiday_matches.is_transferred_original & holiday_matches.type.isin(["Holiday","Additional","Bridge","Transfer"]))
holiday_matches["is_event"]=holiday_matches.type.eq("Event") & ~holiday_matches.is_transferred_original
holiday_matches["is_work_day"]=holiday_matches.type.eq("Work Day")
holiday_matches["transferred_original_count"]=holiday_matches.is_transferred_original.astype("int8")
holiday_matches["active_holiday_count"]=holiday_matches.is_active_holiday.astype("int8")
holiday_matches["transfer_date_count"]=holiday_matches.is_transfer_date.astype("int8")
holiday_matches["event_count"]=holiday_matches.is_event.astype("int8")
holiday_matches["work_day_count"]=holiday_matches.is_work_day.astype("int8")
for loc in ["National","Regional","Local"]:
    holiday_matches[f"active_locale_{loc.lower()}_count"]=(holiday_matches.is_active_holiday & holiday_matches.locale.eq(loc)).astype("int8")
    holiday_matches[f"matched_locale_{loc.lower()}_record_count"]=holiday_matches.locale.eq(loc).astype("int8")
for event_type in sorted(holiday_matches.type.dropna().unique()):
    slug=re.sub(r"[^a-z0-9]+","_",str(event_type).lower()).strip("_")
    holiday_matches[f"matched_type_{slug}_record_count"]=holiday_matches.type.eq(event_type).astype("int8")
for event_type in ["Holiday","Additional","Bridge","Transfer"]:
    slug=event_type.lower()
    holiday_matches[f"active_type_{slug}_count"]=(holiday_matches.is_active_holiday & holiday_matches.type.eq(event_type)).astype("int8")
sum_cols=[c for c in holiday_matches.columns if c.endswith("_count") and c not in {"transferred_original_count","active_holiday_count","transfer_date_count","event_count","work_day_count"}]
holiday_store=holiday_matches.groupby(["date","store_nbr"],as_index=False).agg(
    matched_event_count=("description","size"),transferred_original_count=("transferred_original_count","sum"),
    active_holiday_count=("active_holiday_count","sum"),transfer_date_count=("transfer_date_count","sum"),
    event_count=("event_count","sum"),work_day_count=("work_day_count","sum"),
    active_holiday_types=("type",lambda x:" | ".join(sorted(set(holiday_matches.loc[x.index].loc[holiday_matches.loc[x.index,"is_active_holiday"],"type"].dropna().astype(str))))),
    matched_event_types=("type",lambda x:" | ".join(sorted(set(x.dropna().astype(str))))),
    matched_locales=("locale",lambda x:" | ".join(sorted(set(x.dropna().astype(str))))),
    **{c:(c,"sum") for c in sum_cols})
assert not holiday_store.duplicated(["date","store_nbr"]).any()
print("Matched source event rows:",len(holiday_matches),"Aggregated Date-Store rows:",len(holiday_store))
display(holiday_store.head())


Matched source event rows: 9950 Aggregated Date-Store rows: 9601


,date,store_nbr,matched_event_count,transferred_original_count,active_holiday_count,transfer_date_count,event_count,work_day_count,active_holiday_types,matched_event_types,matched_locales,active_locale_national_count,matched_locale_national_record_count,active_locale_regional_count,matched_locale_regional_record_count,active_locale_local_count,matched_locale_local_record_count,matched_type_additional_record_count,matched_type_bridge_record_count,matched_type_event_record_count,matched_type_holiday_record_count,matched_type_transfer_record_count,matched_type_work_day_record_count,active_type_holiday_count,active_type_additional_count,active_type_bridge_count,active_type_transfer_count
0,2012-03-02,52,1,0,1,0,0,0,Holiday,Holiday,Local,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0,0
1,2012-03-02,53,1,0,1,0,0,0,Holiday,Holiday,Local,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0,0
2,2012-04-01,12,1,0,1,0,0,0,Holiday,Holiday,Regional,0,0,1,1,0,0,0,0,0,1,0,0,1,0,0,0
3,2012-04-01,13,1,0,1,0,0,0,Holiday,Holiday,Regional,0,0,1,1,0,0,0,0,0,1,0,0,1,0,0,0
4,2012-04-12,37,1,0,1,0,0,0,Holiday,Holiday,Local,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0,0


## 7. Ghép holiday features và kiểm tra cardinality

Các bộ đếm `active_*` mô tả ngày nghỉ hoạt động; `matched_*_record_count` chỉ đếm bản ghi đã khớp theo locale, kể cả bản ghi ngày gốc transferred. Ngày không có event khớp nhận count bằng 0; chuỗi loại active holiday để rỗng.


In [8]:
holiday_count_cols=[c for c in holiday_store.columns if c.endswith("_count")]
train_base=train_base.merge(holiday_store,on=["date","store_nbr"],how="left",validate="many_to_one",sort=False)
test_base=test_base.merge(holiday_store,on=["date","store_nbr"],how="left",validate="many_to_one",sort=False)
for frame in (train_base,test_base):
    frame[holiday_count_cols]=frame[holiday_count_cols].fillna(0).astype("int32")
    frame[["active_holiday_types","matched_event_types","matched_locales"]]=frame[["active_holiday_types","matched_event_types","matched_locales"]].fillna("")
assert len(train_base)==train_rows_before and len(test_base)==test_rows_before
assert not train_base.duplicated(["date","store_nbr","family"]).any()
assert not test_base.duplicated(["date","store_nbr","family"]).any()
print("Final base row counts preserved:",len(train_base),len(test_base))


Final base row counts preserved: 3000888 28512


## 8. Giữ transactions lịch sử riêng

Transactions là số giao dịch trong chính ngày đó, nên chưa biết tại thời điểm dự báo sales cùng ngày. Không ghép cột này trực tiếp vào bảng modeling. Lưu bảng lịch sử riêng để bước feature engineering có thể tạo lag/rolling đúng thời điểm hoặc dự báo transactions trước.


In [9]:
transactions_history=transactions.copy()
transaction_coverage=pd.DataFrame([{"rows":len(transactions_history),"start_date":transactions_history.date.min(),"end_date":transactions_history.date.max(),"unique_stores":transactions_history.store_nbr.nunique(),"unique_dates":transactions_history.date.nunique(),"store_date_key_unique":not transactions_history.duplicated(["date","store_nbr"]).any(),"test_period_actual_transactions_available":bool(transactions_history.date.max()>=test.date.min())}])
display(transaction_coverage)


,rows,start_date,end_date,unique_stores,unique_dates,store_date_key_unique,test_period_actual_transactions_available
0,83488,2013-01-01,2017-08-15,54,1682,True,False


## 9. Kiểm tra integrity và lưu dữ liệu processed

Lưu CSV dẫn xuất vào `data/processed/`, không ghi đè raw. Oil missing được giữ; transactions nằm ở sidecar riêng.


In [ ]:
integrity_checks=pd.DataFrame([
 {"check":"train_row_count_preserved","passed":len(train_base)==train_rows_before},
 {"check":"test_row_count_preserved","passed":len(test_base)==test_rows_before},
 {"check":"train_business_key_unique","passed":not train_base.duplicated(["date","store_nbr","family"]).any()},
 {"check":"test_business_key_unique","passed":not test_base.duplicated(["date","store_nbr","family"]).any()},
 {"check":"train_series_count_preserved","passed":train_base[["store_nbr","family"]].drop_duplicates().shape[0]==series_n},
 {"check":"holiday_date_store_unique","passed":not holiday_store.duplicated(["date","store_nbr"]).any()},
 {"check":"oil_missing_retained","passed":train_base.dcoilwtico.isna().sum()>0},
 {"check":"transactions_not_joined_as_same_day_feature","passed":"transactions" not in train_base.columns}])
display(integrity_checks)
assert integrity_checks.passed.all(),"A check failed; inspect before saving."
date_coverage_report.to_csv(REPORT_DIR/"date_coverage.csv",index=False)
oil_coverage_report.to_csv(REPORT_DIR/"oil_coverage.csv",index=False)
transaction_coverage.to_csv(REPORT_DIR/"transaction_coverage.csv",index=False)
integrity_checks.to_csv(REPORT_DIR/"integrity_checks.csv",index=False)
train_base.sort_values("id").to_csv(PROCESSED_DIR/"train_processed.csv",index=False)
test_base.sort_values("id").to_csv(PROCESSED_DIR/"test_processed.csv",index=False)
transactions_history.to_csv(PROCESSED_DIR/"transactions_history.csv",index=False)
print("Saved processed datasets:",PROCESSED_DIR)
print("Saved reports:",REPORT_DIR)


,check,passed
0,train_row_count_preserved,True
1,test_row_count_preserved,True
2,train_business_key_unique,True
3,test_business_key_unique,True
4,train_series_count_preserved,True
5,holiday_date_store_unique,True
6,oil_missing_retained,True
7,transactions_not_joined_as_same_day_feature,True


Saved processed datasets: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\data\processed
Saved reports: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\preprocessing


: 

## 10. Kết luận và giới hạn

### 10.1. Các quyết định preprocessing

- Bốn ngày 25/12 (2013–2016) không xuất hiện trong train được giữ trong báo cáo coverage; không tự tạo dòng hoặc target `sales = 0`.
- Oil missing không bị điền. Biến `oil_date_record_present` được sử dụng để phân biệt ngày không có bản ghi nguồn với ngày có bản ghi nhưng giá dầu bị thiếu.
- Holiday được ghép theo phạm vi địa lý: National, Regional và Local. Các bản ghi được tổng hợp ở cấp Date-Store trước khi merge để tránh nhân số dòng.
- Ngày lễ có `transferred=True` được tách khỏi ngày tổ chức thực tế (`type="Transfer"`). Event và Work Day được biểu diễn bằng các biến riêng.
- Transactions cùng ngày không được merge trực tiếp vào bảng modeling. Bảng lịch sử được lưu riêng để tạo lag/rolling features hoặc dự báo transactions trong bước tiếp theo.
- Không tự động loại bỏ outlier hoặc Store-Family series có tỷ lệ zero sales cao vì chưa có đủ bằng chứng để xác định chúng là dữ liệu lỗi.

### 10.2. Các giới hạn của dữ liệu và preprocessing

- Dữ liệu oil vẫn còn missing trong cả train và test. Đặc biệt, có những ngày trong giai đoạn test không có bản ghi giá dầu nguồn.
- Transactions chỉ có dữ liệu đến ngày 15/08/2017, do đó không thể sử dụng transactions thực tế trong giai đoạn test.
- Các giá trị missing chưa được imputation; outlier chưa được xử lý.
- Chưa thực hiện encoding, scaling hoặc tạo lag/rolling features.
- Các biến Holiday được tạo theo quy tắc ghép địa lý và phân loại sự kiện; chưa thể kết luận tác động nhân quả của ngày lễ đến sales.
- Các integrity checks xác nhận số dòng, khóa nghiệp vụ và cardinality sau merge, nhưng việc kiểm soát leakage cho các feature thời gian vẫn cần được thực hiện trong validation và modeling.

### 10.3. Định hướng cho giai đoạn Modeling

- Quyết định phương pháp xử lý oil missing dựa trên thông tin có thể biết tại thời điểm dự báo.
- Không sử dụng backward fill để đưa giá dầu tương lai vào quá khứ.
- Xây dựng transactions lag/rolling hoặc mô hình dự báo transactions riêng nếu cần sử dụng biến này.
- Thiết kế lag và rolling features với quy tắc shift phù hợp, tránh sử dụng target của tương lai.
- Giữ nguyên validation protocol khi so sánh các phương pháp preprocessing và mô hình.

**Kết luận:** Notebook đã tạo được các bảng dữ liệu dẫn xuất, bảo toàn cấu trúc train/test và hạn chế các rủi ro leakage trong những bước merge hiện tại. Những quyết định imputation, feature engineering và xử lý outlier sẽ tiếp tục được đánh giá trong các giai đoạn tiếp theo.